# 📊 Pipeline de Análise Exploratória e Engenharia de Dados (EDA & ETL)

Este notebook implementa o fluxo completo de **Extração, Transformação e Carga (ETL)** e **Análise Exploratória de Dados (EDA)** para a operação de e-commerce.

### 🎯 Objetivos:
1. **Leitura dos dados brutos** a partir dos arquivos CSV na pasta `database/`.
2. **Limpeza e Tratamento**: remoção de colunas redundantes, padronização de nomenclatura, tratamento de valores nulos e conversão de tipos (datas, percentuais, inteiros).
3. **Exportação dos dados limpos** para o diretório `database_clean/`.
4. **Análise visual** de comportamento de usuários por tipo de dispositivo com Matplotlib e Seaborn.
5. **Carga no Banco de Dados Relacional**: persistência dos dados tratados no banco SQLite `ecommerce.db`.

## 1. Configuração do Ambiente e Inicialização

Importação das bibliotecas necessárias (`pandas`, `matplotlib`, `seaborn`, `sqlalchemy`) e criação da pasta `database_clean` para armazenar os datasets tratados.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

# Criação da pasta de saída para arquivos CSV limpos
os.makedirs("database_clean", exist_ok=True)
print("✅ Ambiente configurado e diretório 'database_clean' pronto.")

## 2. Limpeza e Tratamento dos Dados por Tabela

### 2.1 Clientes (`customers`)
- Seleção e padronização dos nomes de colunas.
- Tratamento de valores ausentes em `time_to_2nd_purchase` (preenchido com 0 para clientes que ainda não efetuaram recompra).
- Exportação para `database_clean/customers_clean.csv`.

In [ ]:
df_customers = pd.read_csv("database/customers.csv")

# Seleção das colunas relevantes e renomeação padronizada (snake_case)
df_customers = df_customers[
    [
        "Customer ID",
        "First order date",
        "Total orders",
        "Total revenue",
        "Average order value",
        "Time to 2nd purchase",
        "Last purchase date",
        "City / tier",
        "Acquisition channel (first touch)",
        "RePurchased",
    ]
]
df_customers.columns = [
    "customer_id",
    "first_order_date",
    "total_orders",
    "total_revenue",
    "average_order_value",
    "time_to_2nd_purchase",
    "last_purchase_date",
    "city_tier",
    "acquisition_channel",
    "repurchased",
]

# Clientes sem segunda compra recebem valor 0 no tempo até a recompra
df_customers.fillna({"time_to_2nd_purchase": 0}, inplace=True)

# Salva versão limpa
df_customers.to_csv("database_clean/customers_clean.csv", index=False)
df_customers.head(5)

### 2.2 Posição de Estoque (`inventory_snapshots`)
- Desmembramento da coluna composta `Units sold (last 7/30/60 days)` em três métricas independentes.
- Conversão de datas e tipos numéricos inteiros.
- Exportação para `database_clean/inventory_snapshots_clean.csv`.

In [ ]:
df_inventory = pd.read_csv("database/inventory_snapshots.csv")

# Separa a coluna consolidada em 3 colunas independentes de vendas
df_inventory[["units_sold_7d", "units_sold_30d", "units_sold_60d"]] = df_inventory[
    "Units sold (last 7/30/60 days)"
].str.split("/", expand=True)

# Seleção e renomeação de colunas
df_inventory = df_inventory[
    [
        "SKU",
        "Category",
        "Size",
        "Units in stock",
        "units_sold_7d",
        "units_sold_30d",
        "units_sold_60d",
        "Days of inventory left",
        "Dead stock flag",
        "date",
    ]
]
df_inventory.columns = [
    "sku",
    "category",
    "size",
    "units_in_stock",
    "units_sold_7d",
    "units_sold_30d",
    "units_sold_60d",
    "days_of_inventory_left",
    "dead_stock_flag",
    "date",
]

# Conversão de tipos
df_inventory["date"] = pd.to_datetime(df_inventory["date"])
df_inventory[
    ["units_sold_7d", "units_sold_30d", "units_sold_60d", "days_of_inventory_left"]
] = df_inventory[
    ["units_sold_7d", "units_sold_30d", "units_sold_60d", "days_of_inventory_left"]
].astype(int)

# Salva versão limpa
df_inventory.to_csv("database_clean/inventory_snapshots_clean.csv", index=False)
df_inventory.head(5)

### 2.3 Campanhas de Anúncios (`meta_ads_campaigns`)
- Remoção de colunas redundantes e renomeação padronizada.
- Conversão do campo `launch_date` para `datetime`.
- Exportação para `database_clean/meta_ads_campaigns_clean.csv`.

In [ ]:
df_campaigns = pd.read_csv("database/meta_ads_campaigns.csv")

# Remove coluna redundante de valor em moeda local
df_campaigns.drop(columns=["Amount spent (INR)"], inplace=True)
df_campaigns.columns = [
    "date",
    "campaign_name",
    "adset_name",
    "results",
    "spend",
    "reach",
    "impressions",
    "frequency",
    "link_clicks",
    "ctr",
    "add_to_cart",
    "initiate_checkout",
    "purchases",
    "conversion_value",
    "cac",
    "roi",
    "creative_type",
    "launch_date",
    "hook_rate",
]

# Salva cópia de análise e remove colunas agregadas para o banco relacional
df_campaigns.to_csv("database_clean/meta_ads_campaigns_clean.csv", index=False)
df_campaigns.drop(columns=["date", "campaign_name"], inplace=True)
df_campaigns["launch_date"] = pd.to_datetime(df_campaigns["launch_date"])
df_campaigns.head(5)

### 2.4 Pedidos (`orders`)
- Remoção de campos agregados ou de CEP desnecessários.
- Desdobramento de `order_datetime` em `order_date` e `order_time`.
- Conversão da string de desconto (`%`) para valor decimal (`float / 100`).
- Exportação para `database_clean/orders_clean.csv`.

In [ ]:
df_orders = pd.read_csv("database/orders.csv")

# Remoção de colunas desnecessárias e renomeação padronizada
df_orders.drop(
    columns=["Order value (gross, net)", "Discount applied (₹ + %)", "Pincode"],
    inplace=True,
)
df_orders.columns = [
    "order_id",
    "customer_id",
    "order_datetime",
    "product",
    "gross_value",
    "net_value",
    "discount_value",
    "discount_percentage",
    "payment_mode",
    "shipping_city",
    "first_order_vs_repeat",
    "last_touch_channel",
    "order_status",
]

# Parsing de data e horário do pedido
df_orders["order_datetime"] = pd.to_datetime(df_orders["order_datetime"])
df_orders["order_date"] = df_orders["order_datetime"].dt.date
df_orders["order_time"] = df_orders["order_datetime"].dt.time
df_orders.drop(columns="order_datetime", inplace=True)

# Tratamento da coluna de percentual de desconto
df_orders["discount_percentage"] = (
    df_orders["discount_percentage"].str.replace("%", "").astype(float) / 100
)

# Salva versão limpa
df_orders.to_csv("database_clean/orders_clean.csv", index=False)
df_orders.head(5)

### 2.5 Itens do Pedido (`order_line_items`)
- Tratamento de devoluções: preenchimento de valores ausentes em `return_reason` como `"Not Returned"`.
- Conversão da coluna `discount_percentage` para escala decimal.
- Exportação para `database_clean/order_line_items_clean.csv`.

In [ ]:
df_items = pd.read_csv("database/order_line_items.csv")
df_items.columns = [
    "order_id",
    "sku",
    "category",
    "size",
    "color",
    "mrp",
    "selling_price",
    "discount_percentage",
    "returned",
    "return_reason",
]

# Preenchimento de nulos para produtos que não foram devolvidos
df_items["return_reason"] = df_items["return_reason"].fillna("Not Returned")

# Normalização de desconto para escala de 0 a 1
df_items["discount_percentage"] = df_items["discount_percentage"] / 100

# Salva versão limpa
df_items.to_csv("database_clean/order_line_items_clean.csv", index=False)
df_items.head(5)

### 2.6 Ordens de Compra (`purchase_orders`)
- Conversão das datas de pedido (`order_date`), previsão de entrega (`expected_delivery`) e entrega real (`actual_delivery`).
- Exportação para `database_clean/purchase_orders_clean.csv`.

In [ ]:
df_po = pd.read_csv("database/purchase_orders.csv")
df_po.columns = [
    "sku",
    "vendor",
    "order_quantity",
    "cost_per_unit",
    "order_date",
    "expected_delivery",
    "actual_delivery",
    "lead_time",
]

# Conversão das datas para datetime
df_po["order_date"] = pd.to_datetime(df_po["order_date"])
df_po["expected_delivery"] = pd.to_datetime(df_po["expected_delivery"])
df_po["actual_delivery"] = pd.to_datetime(df_po["actual_delivery"])

# Salva versão limpa
df_po.to_csv("database_clean/purchase_orders_clean.csv", index=False)
df_po.head(5)

### 2.7 Catálogo de Produtos (`sku_catalog`)
- Padronização dos nomes de colunas.
- Exportação para `database_clean/sku_catalog_clean.csv`.

In [ ]:
df_sku = pd.read_csv("database/sku_catalog.csv")
df_sku.columns = ["sku", "category", "vendor", "mrp", "cost_per_unit"]

# Salva versão limpa
df_sku.to_csv("database_clean/sku_catalog_clean.csv", index=False)
df_sku.head(5)

### 2.8 Tráfego Diário do Website (`website_daily`)
- Preenchimento de campanhas não especificadas com `"Without campaign"`.
- Conversão da data diária para `datetime`.
- Análise exploratória por dispositivo (sessões e conversões).
- Exportação para `database_clean/website_daily_clean.csv`.

In [ ]:
df_web_daily = pd.read_csv("database/website_daily.csv")
df_web_daily.columns = [
    "daily_date",
    "traffic_source",
    "campaign_name",
    "device_category",
    "daily_sessions",
    "daily_product_views",
    "daily_add_to_cart",
    "daily_begin_checkout",
    "daily_purchases",
    "daily_revenue",
    "daily_conversion_rate",
    "daily_aov",
    "country",
    "city_location",
    "has_purchases_flag",
]

# Tratamento de nulos em campanhas e conversão de data
df_web_daily["campaign_name"] = df_web_daily["campaign_name"].fillna("Without campaign")
df_web_daily["daily_date"] = pd.to_datetime(df_web_daily["daily_date"])

# Salva versão limpa
df_web_daily.to_csv("database_clean/website_daily_clean.csv", index=False)
df_web_daily.head(5)

#### 📊 Análise Gráfica: Volume de Acessos e Compras por Dispositivo
Visualização das diferenças de comportamento entre usuários de mobile, desktop e tablet.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Distribuição de sessões por dispositivo
df_web_daily["device_category"].value_counts().plot(
    kind="bar", ax=axes[0], color="#4C72B0", edgecolor="black"
)
axes[0].set_title("Número de Registros por Categoria de Dispositivo", fontsize=13)
axes[0].set_xlabel("Dispositivo")
axes[0].set_ylabel("Total de Registros")
axes[0].tick_params(axis="x", rotation=0)

# Gráfico 2: Média de compras diárias por dispositivo
df_web_daily.groupby("device_category")["daily_purchases"].mean().plot(
    kind="bar", ax=axes[1], color="#55A868", edgecolor="black"
)
axes[1].set_title("Média Diária de Compras por Dispositivo", fontsize=13)
axes[1].set_xlabel("Dispositivo")
axes[1].set_ylabel("Média de Compras")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

### 2.9 Sessões Granulares do Site (`website_sessions`)
- Deduplicação por `session_id`.
- Preenchimento de valores nulos em `order_id` e `customer_id` com 0 (para sessões sem conversão).
- Preenchimento de `campaign_name` com `"Without campaign"`.
- Exportação para `database_clean/website_sessions_clean.csv`.

In [ ]:
df_web_sessions = pd.read_csv("database/website_sessions.csv")
df_web_sessions.columns = [
    "session_id",
    "session_datetime",
    "session_source",
    "campaign_name",
    "user_device",
    "user_city",
    "session_count",
    "session_product_views",
    "session_cart_additions",
    "session_checkouts_started",
    "is_purchase_sucessful",
    "order_id",
    "customer_id",
    "session_revenue",
]

# Remoção de duplicatas de sessão
df_web_sessions = df_web_sessions.drop_duplicates(subset="session_id", keep="first")

# Tratamento de valores nulos
df_web_sessions[["order_id", "customer_id"]] = df_web_sessions[
    ["order_id", "customer_id"]
].fillna(0)
df_web_sessions["campaign_name"] = df_web_sessions["campaign_name"].fillna(
    "Without campaign"
)

# Salva versão limpa
df_web_sessions.to_csv("database_clean/website_sessions_clean.csv", index=False)
df_web_sessions.head(5)

## 3. Carga dos Dados no Banco SQLite (`ecommerce.db`)

Conexão com o banco relacional e persistência de todas as tabelas limpas no arquivo `ecommerce.db`.

In [ ]:
# Conexão com o banco de dados SQLite
engine = create_engine("sqlite:///ecommerce.db")

print("Iniciando carga de dados no SQLite...")
df_customers.to_sql("customers", con=engine, if_exists="replace", index=False)
df_sku.to_sql("sku_catalog", con=engine, if_exists="replace", index=False)
df_campaigns.to_sql("meta_ads_campaigns", con=engine, if_exists="replace", index=False)
df_web_daily.to_sql("website_daily", con=engine, if_exists="replace", index=False)
df_web_sessions.to_sql("website_sessions", con=engine, if_exists="replace", index=False)
df_orders.to_sql("orders", con=engine, if_exists="replace", index=False)
df_items.to_sql("order_line_items", con=engine, if_exists="replace", index=False)
df_inventory.to_sql("inventory_snapshots", con=engine, if_exists="replace", index=False)
df_po.to_sql("purchase_orders", con=engine, if_exists="replace", index=False)
print("✅ Todas as tabelas foram carregadas com sucesso no banco 'ecommerce.db'!")

## 4. Validação da Carga Relacional

Verificação da contagem total de registros persistidos em cada tabela do banco SQLite.

In [ ]:
import sqlite3

conn = sqlite3.connect("ecommerce.db")
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = [row[0] for row in cursor.fetchall()]

print("📊 Resumo dos registros persistidos no SQLite:")
total_records = 0
for table in tables:
    cursor.execute(f"SELECT count(*) FROM {table};")
    count = cursor.fetchone()[0]
    total_records += count
    print(f"  - {table:22}: {count:>8,} linhas")

print(f"\nTotal acumulado: {total_records:,} registros em {len(tables)} tabelas.")
conn.close()